# Mandatory Disclosure and Housing Market Dynamics
## French Housing Data — Structural Estimation — Patras (2024)

This notebook estimates the structural model of mandatory EPC disclosure on real French housing transaction data. Buyers are heterogeneous in their EPC sensitivity $\theta$; a fraction $\pi$ are *informed* and adjust their willingness-to-pay for the disclosed rating. Informed sellers with an unfavorable EPC rating may withhold their certificate, incurring a hiding cost $c_0$. The two key structural parameters recovered from the data are:

| Parameter | Symbol | Empirical interpretation |
|-----------|--------|-------------------------|
| Buyer EPC sensitivity share | $\rho$ | Share of buyers who price in the EPC label — estimated at ~74%, indicating strong but incomplete market discipline |
| Hiding cost | $c_0$ | Per-unit cost of concealing the EPC label — calibrates the policy subsidy required to flip the equilibrium toward full disclosure |

In [1]:
import logging
import warnings
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s")

from src.viz import (
    chart_data_overview, chart_hedonic_params, chart_hiding_premiums,
    chart_moment_fit, chart_estimates_by_market, chart_equilibrium_objects,
    chart_msm_surface, chart_belief_function, chart_market_scatter,
    chart_renovation_thresholds,
    chart_event_study_ts,
    chart_did_estimates,
)

from src.data_prep   import load_listings, merge_datasets, classify_listings, compute_market_stats
from src.hedonic     import (estimate_hedonic, calibrate_demand_scale,
                              calibrate_demand_scale_m, compute_hiding_premium,
                              estimate_rho_direct)
from src.moments     import extract_data_moments, build_weighting_matrix
from src.estimation  import (build_valuation_params, estimate_msm, msm_objective,
                              compute_market_subsidies)
from src.model       import solve_equilibrium, belief_function
from src.validation  import run_all_validation
from src.event_study import (load_price_indices, compute_did,
                              estimate_rho_c0, aggregate_event_study)


## Configuration

| Parameter | Value | Description |
|-----------|-------|-------------|
| `DATA_PATH` | `data/listings_enriched_APARTMENT.parquet` | Path to enriched listings file |
| `EPC_THRESHOLD` | 4 | DPE grade cutoff: ratings at or below this threshold are classified as low quality ($\theta=0$) |
| `MIN_MARKET_SIZE` | 1000 | Minimum listings per market to be included in estimation |
| `UNVERIFIED_DISCLOSERS` | `"honest"` | Treatment of unverified disclosures: assume truthful |
| `WEIGHTING` | `"equal"` | Moment weighting scheme (equal weights for robustness) |
| `N_RESTARTS` | 5 | Number of random restarts in MSM optimization |
| `C0_INIT` | 0.05 | Initial hiding cost guess for the optimizer |
| `INDEX_PATH` | `data/index_APARTMENT.csv` | Hedonic price index panel (110 markets × 70 months) |
| `MIN_VOLUME` | `20` | Min avg monthly transactions per EPC cell for DiD volume filter |
| `DROP_MOMENTS` | `[0, 1, 4]` | Drop $m_1$, $m_2$ (fit by construction) and $m_5$ (overid test) |

Controls include both physical property characteristics and location variables. Using `city_id` as a fixed effect in the hedonic absorbs local price levels, so $\hat{\beta}_1$ captures the within-city EPC premium.

In [2]:
# --- Data and sample configuration ---
DATA_PATH          = "data/listings_enriched_HOUSE.parquet"
INDEX_PATH         = "data/index_HOUSE.csv"
EPC_THRESHOLD      = 4
MIN_MARKET_SIZE    = 1000
UNVERIFIED_DISCLOSERS = "honest"

# --- Hedonic controls ---
CONTROLS = [
    "built_year", "floor", "rooms",
    "elevator", "cellar", "city_id", "distance",
]

# --- Estimation settings ---
WEIGHTING    = "inverse_variance"
DROP_MOMENTS = [0, 1, 2, 4]  # drop m₁, m₂ (fit by construction), m₃ (selection-contaminated;
                               # DiD first stage handles identification of ρ_m), m₅ (overid test)
N_RESTARTS = 5
C0_INIT    = 0.05
MIN_VOLUME = 20    # min avg monthly transactions per EPC cell for DiD volume filter
START_YEAR = 2022  # drop 2021: only H2 data (reform started July 2021), beta_1 unreliable
SEED       = 42


## Section 0 — French Housing Data

The dataset covers residential apartment listings in French cities enriched with DPE (Diagnostic de Performance Energetique) scores. Each listing is classified into one of four groups based on information status $K$ (whether the seller has an ADEME-verified EPC) and disclosure decision $d$:

- **Verified disclosers** (`truthful_disclosure`, $K=1$, $d=1$) — sellers whose listed EPC matches the ADEME registry. The estimation sample for the hedonic regression.
- **Strategic hiders** (`strategic_hiding`, $K=1$, $d=0$, $\theta=0$) — informed sellers with a poor rating who withheld their certificate.
- **Uninformed sellers** (`uninformed`, $K=0$) — sellers without an EPC in the ADEME database; cannot disclose.
- **Falsifiers** (`falsification`, $K=1$, $d=1$, EPC $\neq$ ADEME) — sellers who declared a rating inconsistent with the registry.

Markets are defined at the functional-urban-area level (`market_id`). Only markets with at least `MIN_MARKET_SIZE` listings are retained for structural estimation to ensure sufficient moment precision.

In [3]:
# Load and classify listings
df_raw = load_listings(DATA_PATH, recode_market_id=True)
df     = merge_datasets(df_raw, ademe=None, epc_threshold=EPC_THRESHOLD)
df     = classify_listings(df, unverified_disclosers=UNVERIFIED_DISCLOSERS)

print(f"Raw listings loaded    : {len(df_raw):,}")
print(f"After classification   : {len(df):,}")
print(f"Disclosure rate        : {df['d'].mean():.1%}")
print(f"Share theta=1 (good EPC, threshold≤{EPC_THRESHOLD}): {df['theta'].mean():.1%}")

# Drop pre-START_YEAR listings (2021 = H2 only; reform started July 2021 → β̂₁ unreliable)
df['start_date'] = pd.to_datetime(df['start_date'])
df = df[df['start_date'].dt.year >= START_YEAR].copy()
print(f"After {START_YEAR}+ filter : {len(df):,} listings ({df['start_date'].dt.year.min()}–{df['start_date'].dt.year.max()})")

Raw listings loaded    : 687,878
After classification   : 687,878
Disclosure rate        : 90.3%
Share theta=1 (good EPC, threshold≤4): 61.6%
After 2022+ filter : 632,575 listings (2022–2024)


In [4]:
# Compute market-level statistics
market_stats = compute_market_stats(df, controls=CONTROLS)

# Filter to markets with sufficient sample size
large_enough      = market_stats["n_total"] >= MIN_MARKET_SIZE
estimable_markets = market_stats.index[large_enough & (market_stats["estimable"] == True)].tolist()

print(f"Total markets          : {len(market_stats)}")
print(f"Estimable markets      : {len(estimable_markets)}")
print(f"Listings in sample     : {market_stats.loc[estimable_markets, 'n_total'].sum():,}")

Total markets          : 674
Estimable markets      : 108
Listings in sample     : 495,763


In [5]:
# Group counts and sample overview
group_counts = df.groupby("group").size().rename("count").to_frame()
group_counts["share"] = group_counts["count"] / len(df)
print(group_counts.to_string())

fig = chart_data_overview(df, market_stats)
fig.show()

                      count     share
group                                
strategic_hiding       7257  0.011472
truthful_disclosure  575825  0.910287
uninformed            49493  0.078241


## Section 2b — Market Heterogeneity

Before estimation, it is worth documenting how markets differ in their observable characteristics. Variation in market size, average price, and disclosure rates is what drives cross-market identification. The scatter chart plots each market as a point in the space of hiding rate vs. average price (or a related two-dimensional summary), revealing whether larger or more expensive markets tend to have higher or lower disclosure rates.

In [6]:
fig = chart_market_scatter(market_stats.loc[estimable_markets])
fig.show()

In [7]:
market_stats.loc[estimable_markets].describe().T.round(4)

,count,mean,std,min,25%,50%,75%,max
n_total,108.0,4.590398e+03,1.480402e+04,1025.0000,1552.7500,2168.5000,3.720500e+03,1.537490e+05
n_informed,108.0,1.953482e+03,5.872835e+03,324.0000,666.2500,1002.5000,1.594000e+03,6.090200e+04
n_informed_low,108.0,7.596389e+02,2.730847e+03,103.0000,249.7500,379.5000,5.690000e+02,2.854300e+04
n_informed_high,108.0,1.193843e+03,3.169912e+03,189.0000,400.7500,579.5000,9.172500e+02,3.235900e+04
pi_hat,108.0,4.403000e-01,5.170000e-02,0.3108,0.4142,0.4391,4.779000e-01,5.538000e-01
theta_bar_hat,108.0,6.155000e-01,1.049000e-01,0.3742,0.5383,0.6037,6.801000e-01,8.617000e-01
sigma_0_hat,108.0,8.100000e-02,7.770000e-02,0.0080,0.0363,0.0573,8.530000e-02,4.571000e-01
sigma_1_hat,108.0,0.000000e+00,0.000000e+00,0.0000,0.0000,0.0000,0.000000e+00,0.000000e+00
p_bar_d1_th1,108.0,7.735700e+00,2.990000e-01,7.1914,7.5189,7.7053,7.930100e+00,8.538200e+00
p_bar_d1_th0,108.0,7.597200e+00,3.272000e-01,6.9705,7.3736,7.5673,7.783500e+00,8.440300e+00


## Section 2 — Hedonic Regression

### Equation (110)
$$\log p_i = \alpha_m + \beta_1 \theta_i + X_i' \gamma + \varepsilon_i$$

The hedonic regression is estimated **only on truthful disclosers**. Restricting to this sub-sample avoids selection bias from hiding.

**Interpreting $\hat{\beta}_1$ in the data.** The estimated coefficient is approximately $\hat{\beta}_1 \approx 0.037$, meaning a one-unit improvement in the EPC grade is associated with a **3.7% price premium**. This is much smaller than the policy benchmark of ~6% used in some subsidy designs (Fuerst et al.), consistent with:
- Attenuation bias from uninformed buyers: $(1 - \rho\pi)$ fraction of buyers do not price the EPC label.
- Selection: voluntary disclosers are positively selected on quality, compressing the within-group gradient.

**Demand scale calibration.** The raw $\hat{\beta}_1$ is rescaled to recover the structural willingness-to-pay gradient for informed buyers ($\rho\pi$ of the population).

In [8]:
# Estimate hedonic model on truthful disclosers only
hedonic_params = estimate_hedonic(
    df[df["market_id"].isin(estimable_markets)],
    market_col="market_id",
    controls=CONTROLS,
)

# Calibrate demand scales
scale   = calibrate_demand_scale(hedonic_params)
scale_m = calibrate_demand_scale_m(hedonic_params)

print(f"Estimated beta_hat_1 : {hedonic_params.beta_1:.4f}  (~3.7% EPC premium per grade)")
print(f"Pooled demand scale  : {scale:.4f}")
print(f"Hedonic R-squared    : {hedonic_params.fitted_model.rsquared:.4f}")
print(f"sigma_eps            : {hedonic_params.sigma_eps:.4f}")
print(f"N observations       : {hedonic_params.n_obs:,}")

INFO | [hedonic] β̂₁ = 0.0615  (pooled EPC premium, log-price)


INFO | [hedonic] σ̂_ε = 0.2758  (residual std dev → demand scale)


INFO | [hedonic] n = 205256, R² = 0.616


INFO | [hedonic] β̂₁_m (market-specific): mean=0.1053  std=0.0449  [0.0095, 0.2611]  (5 markets used pooled fallback: too few obs / no EPC variation / p>0.10)


INFO | [scale] σ̂_ε = 0.2758 → logistic scale = 0.5002


INFO | [scale_m] market-specific logistic scales: mean=0.4687  std=0.0573  [0.3647, 0.6105]


Estimated beta_hat_1 : 0.0615  (~3.7% EPC premium per grade)
Pooled demand scale  : 0.5002
Hedonic R-squared    : 0.6156
sigma_eps            : 0.2758
N observations       : 205,256


In [9]:
fig = chart_hedonic_params(
    hedonic_params,
    market_stats.loc[estimable_markets],
)
fig.show()

### Direct $\rho$ Estimator

The market-level hiding premium $\hat{\Delta}_m$ — the average log price gap between hidden and disclosed listings after controlling for observable quality — provides a closed-form first-stage estimate of $\hat{\rho}^{\text{dir}}_m$. In the real data, a positive and significant $\hat{\Delta}_m$ confirms that hidden listings transact at a premium, consistent with adverse selection: sellers who hide tend to have EPC ratings only slightly below $\bar{\theta}$, so buyers rationally bid close to the disclosed price.

In [10]:
# Compute hiding premium and direct rho estimator
df_est = df[df["market_id"].isin(estimable_markets)]

df_hiders, market_premium = compute_hiding_premium(df_est, hedonic_params)
rho_direct_df = estimate_rho_direct(
    market_stats.loc[estimable_markets],
    market_premium,
    hedonic_params,
)

print(f"Mean hiding premium delta_bar_m : {rho_direct_df['delta_bar'].mean():.4f}")
print(f"Mean rho_hat (direct, clipped)  : {rho_direct_df['rho_hat_clipped'].mean():.4f}")
print(f"  -> ~{rho_direct_df['rho_hat_clipped'].mean():.0%} of buyers are EPC-sensitive (first-stage)")

fig = chart_hiding_premiums(rho_direct_df)
fig.show()

INFO | [hiding premium] 5720 strategic hiders across 108 markets


INFO | [hiding premium] Mean Δ̄ = 0.0310


INFO | [hiding premium] Std  Δ̄ = 0.1169


WARNING | 87 markets have ρ̂_m outside [0,1]. Check sample size, hedonic specification, or approximation quality.


INFO | [direct ρ] Mean ρ̂ = 0.7544


INFO | [direct ρ] Std  ρ̂ = 3.0449


Mean hiding premium delta_bar_m : 0.0310
Mean rho_hat (direct, clipped)  : 0.5225
  -> ~52% of buyers are EPC-sensitive (first-stage)


## Section 1b — DiD Event Study Identification

Before running MSM, we use the panel of hedonic price indices to identify $\hat{\rho}_m$ and $\hat{c}_{0m}$ directly via a difference-in-differences strategy around the **July 2021 DPE reform**.

Pre-reform (≤ 2021-06): the DPE was not legally opposable — buyers discounted EPC scores, so $\rho_m \approx 0$.
Post-reform (≥ 2021-07): the DPE became legally binding — $\rho_m > 0$, varying across markets.

$$\hat{\beta}^{1}_m = \underbrace{(\bar{p}_{ABC} - \bar{p}_{EFG})_{\text{post}}}_{\text{EPC premium post}} - \underbrace{(\bar{p}_{ABC} - \bar{p}_{EFG})_{\text{pre}}}_{\text{EPC premium pre}}$$

$$\hat{\Delta}_m = (\bar{p}_{\varnothing} - \bar{p}_{EFG})_{\text{post}} - (\bar{p}_{\varnothing} - \bar{p}_{EFG})_{\text{pre}}$$

Inverting the equilibrium condition $\Delta_m = \rho_m \cdot \hat{\beta}^{1}_m \cdot e^*(\hat{\sigma}_{0m})$ gives:

$$\hat{\rho}_m^{\text{DiD}} = \frac{\hat{\Delta}_m}{\hat{\beta}^{1}_m \cdot \hat{e}^*_m}, \qquad \hat{c}_{0m}^{\text{DiD}} = \hat{\Delta}_m$$

These DiD estimates are then used as the **warm-start** for MSM, substantially reducing the risk of convergence to local optima.

**Note on the structural $\beta_{1,m}$.** The hedonic regression gives $\hat{\beta}^{\text{hed}}_{1,m} \approx 0.056$ (mean), but this includes pre-existing EPC-correlated price differences (energy-efficient apartments tend to be newer and better-located independently of buyer EPC sensitivity). The reform-identified $\hat{\beta}^{\text{DiD}}_{1,m} \approx 0.017$ strips these out by differencing. We use $\hat{\beta}^{\text{DiD}}_{1,m}$ as the structural $\beta_{1,m}$ in `ValuationParams` for DiD-covered markets — it enters the equilibrium condition $\Delta(e^*;\rho) = \rho \cdot \beta_{1,m} \cdot e^*$, the model moment $m_3^{\text{model}} = \beta_{1,m} \cdot e^*$, and the DiD estimator $\hat{\rho}_m = \hat{\Delta}_m / (\beta_{1,m} \cdot \hat{e}^*_m)$, ensuring all three are on the same scale. Markets without DiD coverage fall back to $\hat{\beta}^{\text{hed}}_{1,m}$.

In [11]:
# Load price index panel and compute market-level DiD estimates
index_df  = load_price_indices(INDEX_PATH)
did_est   = compute_did(index_df, min_volume=MIN_VOLUME)

# Structural identification: ρ̂_m and ĉ₀_m from DiD
es_params = estimate_rho_c0(did_est, market_stats)

print(f"DiD markets identified : {es_params.rho_m.notna().sum()}")
print(f"Mean ρ̂_m  (DiD)       : {es_params.rho_m.mean():.4f}")
print(f"Std  ρ̂_m  (DiD)       : {es_params.rho_m.std():.4f}")
print(f"Mean ĉ₀_m (DiD)       : {es_params.c0_m.mean():.4f}")
print(f"Std  ĉ₀_m (DiD)       : {es_params.c0_m.std():.4f}")

# Aggregate event study time series
es_ts = aggregate_event_study(index_df)

es_ts[["k","epc_premium","nondiscl_premium","n_markets"]].round(4)
# Build DiD_delta dict for moment replacement: raw DiD_Δ_m (unclipped)
# Keys are market_ids that match listings data; NaN for unidentified markets
did_delta_dict = did_est.did["DiD_delta"].to_dict()
# Exclude markets where DiD_beta1 <= 0 (ρ not identified → Δ_m unreliable)
did_delta_dict = {
    m: v for m, v in did_delta_dict.items()
    if did_est.did.loc[m, "DiD_beta1"] > 0
}
print(f"DiD delta available for {len(did_delta_dict)} markets")

# DiD_beta1_m: reform-induced EPC quality premium (from price index)
# Used to replace hedonic beta_1 in ValuationParams for DiD-covered markets,
# so that the structural beta_1 entering the model is on the same scale as
# the DiD_delta moment (both reform-identified, not contaminated by correlated quality).
did_beta1_dict = {
    m: float(v) for m, v in did_est.did["DiD_beta1"].items()
    if v > 0  # only markets where DiD_beta1 is identified and positive
}
print(f"DiD_beta1 available for {len(did_beta1_dict)} markets  "
      f"(mean={sum(did_beta1_dict.values())/len(did_beta1_dict):.4f})")


INFO | [event_study] Loaded price indices: 120 markets × ~70 months  (2018-10-31 → 2024-07-31)


INFO | [event_study] Reform date: 2021-07-01  |  Pre months: 33  Post months: 37


INFO | [event_study] Volume filter (pre, min=20): dropped 5 thin markets, 115 remain.


INFO | [event_study] Volume filter (post, min=20): dropped 2 thin markets, 113 remain.


WARNING | [event_study] 6/113 markets have DiD_beta1 ≤ 0 (ABC appreciated less than EFG post-reform). ρ_m will be set to NaN for these markets.


...

WARNING | [event_study] 8 markets in index data have no match in market_stats (listings data). Proceeding with 105 matched markets.


WARNING | [event_study] 73 markets have ρ̂_m outside [0,1] before clipping — check DiD estimates for these markets.


INFO | [event_study] ρ̂_m identified for 100/105 markets  mean=0.638  std=0.432  [0.000, 1.000]


INFO | [event_study] ĉ₀_m: mean=0.0397  std=0.0384  [0.0000, 0.1612]


DiD markets identified : 100
Mean ρ̂_m  (DiD)       : 0.6378
Std  ρ̂_m  (DiD)       : 0.4316
Mean ĉ₀_m (DiD)       : 0.0397
Std  ĉ₀_m (DiD)       : 0.0384
DiD delta available for 107 markets
DiD_beta1 available for 107 markets  (mean=0.0621)


In [12]:
# Event study time series: aggregate price gaps around the reform
fig = chart_event_study_ts(es_ts)
fig.show()

In [13]:
# DiD structural estimates by market: ρ̂_m^DiD and ĉ₀_m
fig = chart_did_estimates(es_params)
fig.show()

## Section 3 — MSM Estimation

### Moments

One moment is matched per market $m$ in the MSM objective.
Moments $m_1$, $m_2$, $m_3$, and $m_5$ are excluded: $m_1$ and $m_2$ are fit by
construction after residualisation; $m_3$ is excluded because the cross-sectional
non-discloser price is contaminated by unobservable selection into hiding — the DiD
first stage handles this via index differencing (ESTIMATION\_STRATEGY.md, Sec. 5,
change \#8); $m_5$ is an overidentifying restriction tested separately.

| Moment | Data | Model | Identifies | Status |
|--------|------|-------|------------|--------|
| *(excl.)* $m_3$ | $\hat{\Delta}_m^{\text{DiD}}$ | $\rho_m \cdot \beta^1_m \cdot e^*(\sigma_0^*)$ | $\rho_m$ | diagnostic only |
| $m_4$ | $\hat{\sigma}_{0,m}$ (hiding rate, informed low-quality) | $\sigma_0^*(\rho_m, c_0)$ | $c_{0m}$ | **active** |
| *(overid)* $m_5$ | $\hat{\sigma}_{1,m} \approx 0$ | $0$ (Corollary) | test only | excluded |

$m_3$ is computed and reported as a diagnostic: comparing $\hat{\Delta}_m^{\text{DiD}}$
to the model-implied $\rho_m \cdot \beta^1_m \cdot e^*$ provides a specification check
of the structural equation (ESTIMATION\_STRATEGY.md Sec. 1: $\Delta_m = \rho_m \cdot \beta_{1,m} \cdot e^*$).
The structural $\beta_{1,m}$ uses the hedonic estimate (full post-reform EPC premium),
which enters buyer valuations and the equilibrium solver.

### Warm-start

The MSM optimizer is warm-started from $\hat{\rho}_m^{\text{DiD}}$ where available,
and from the (clipped) direct hedonic estimator otherwise.


In [14]:
# Extract empirical moments per market
data_moments = extract_data_moments(
    df[df["market_id"].isin(estimable_markets)],
    market_col="market_id",
    hedonic_params=hedonic_params,
    did_delta=did_delta_dict,
)
data_moments = {k: v for k, v in data_moments.items() if k in estimable_markets}

# Build structural valuation parameters per market
# Hedonic β̂₁_m is used for the equilibrium solver (full post-reform EPC premium).
# did_beta1 is NOT passed: the DiD β₁ is only used in the direct estimator (first stage).
market_params = {}
for mid in estimable_markets:
    vp = build_valuation_params(
        market_id=mid,
        market_stats=market_stats,
        hedonic_params=hedonic_params,
        scale=scale,
        scale_m=scale_m,
    )
    market_params[mid] = vp

# Warm-start ρ from DiD estimates; fall back to direct estimator if DiD unavailable.
# Clip to [0.05, 0.95]: starting at exactly 0 or 1 puts the equilibrium solver in a
# degenerate regime (full unraveling / partial separation) where the gradient of
# m₄_model w.r.t. (ρ, c₀) is flat, trapping the optimizer at the corner.
RHO_INIT_BOUNDS = (0.05, 0.80)  # capped at 0.80: DiD mean≈0.64 for houses, 0.95 drives corners
rho_did   = es_params.rho_m.to_dict()
rho_dir   = rho_direct_df["rho_hat_clipped"].to_dict()
rho_init  = {
    m: float(np.clip(
        rho_did[m] if m in rho_did and not np.isnan(rho_did[m])
        else rho_dir.get(m, 0.5),
        *RHO_INIT_BOUNDS
    ))
    for m in data_moments
}

# Warm-start c₀ from DiD_Δ_m where available (upper bound on c₀, but much better
# than a flat C0_INIT for markets whose optimizer was stuck at partial_separation).
# DiD_Δ_m is clipped at C0_INIT/2 from below to avoid starting too close to zero.
c0_init_dict = {
    m: float(np.clip(did_delta_dict[m], C0_INIT / 2, None))
    if m in did_delta_dict
    else C0_INIT
    for m in data_moments
}

n_did_warmstart = sum(
    1 for m in data_moments
    if m in rho_did and not np.isnan(rho_did.get(m, float("nan")))
)
n_c0_did = sum(1 for m in data_moments if m in did_delta_dict)
print(f"Markets entering MSM       : {len(estimable_markets)}")
print(f"  ρ warm-started by DiD    : {n_did_warmstart}")
print(f"  ρ warm-started by direct : {len(estimable_markets) - n_did_warmstart}")
print(f"  c₀ warm-started by DiD   : {n_c0_did}")
print(f"  c₀ warm-started by C0_INIT: {len(estimable_markets) - n_c0_did}")
print(f"Mean rho_init   : {np.mean(list(rho_init.values())):.4f}")
print(f"Mean c0_init    : {np.mean(list(c0_init_dict.values())):.4f}")


Markets entering MSM       : 108
  ρ warm-started by DiD    : 84
  ρ warm-started by direct : 24
  c₀ warm-started by DiD   : 84
  c₀ warm-started by C0_INIT: 24
Mean rho_init   : 0.4960
Mean c0_init    : 0.0474


In [15]:
# --- MSM feasibility pre-screen ---
# Some markets have no mixed-strategy equilibrium for any (ρ, c₀) in the feasible
# parameter space given their observable characteristics (π̂_m, θ̄_m, β̂₁_m).
# Running MSM on these markets produces corner solutions (full_unraveling or
# partial_separation) that invalidate standard errors and moment-fit diagnostics.
# We screen them out before estimation and report them separately.
#
# Feasibility criterion: ∃ ρ ∈ [0.05, 0.95], c₀ ∈ {0.005, 0.01, 0.025, 0.05}
#   such that solve_equilibrium(ρ, c₀, π̂_m, θ̄_m, params_m).eq_type == "mixed_strategy"

from src.model import solve_equilibrium as _solve_eq

def _is_msm_feasible(mid, market_stats, market_params,
                     rho_grid=None, c0_grid=None):
    if rho_grid is None:
        rho_grid = np.linspace(0.05, 0.95, 10)
    if c0_grid is None:
        c0_grid = [0.005, 0.01, 0.025, 0.05]
    row    = market_stats.loc[mid]
    params = market_params[mid]
    pi, theta_bar = float(row["pi_hat"]), float(row["theta_bar_hat"])
    for rho in rho_grid:
        for c0 in c0_grid:
            eq = _solve_eq(rho, c0, pi, theta_bar, params)
            if eq.eq_type == "mixed_strategy":
                return True
    return False

msm_feasible   = [m for m in estimable_markets if _is_msm_feasible(m, market_stats, market_params)]
msm_infeasible = [m for m in estimable_markets if m not in msm_feasible]

print(f"MSM-feasible markets   : {len(msm_feasible)} / {len(estimable_markets)}")
print(f"MSM-infeasible (dropped): {len(msm_infeasible)}")
if msm_infeasible:
    rows = []
    for m in msm_infeasible:
        r = market_stats.loc[m]
        rows.append({
            "market_id"  : m,
            "pi_hat"     : round(float(r["pi_hat"]), 3),
            "theta_bar"  : round(float(r["theta_bar_hat"]), 3),
            "sigma_0"    : round(float(r["sigma_0_hat"]), 3),
            "n_total"    : int(r["n_total"]),
        })
    print(pd.DataFrame(rows).set_index("market_id").to_string())

# Filter moments and params to feasible markets only
data_moments  = {m: v for m, v in data_moments.items()  if m in msm_feasible}
market_params = {m: v for m, v in market_params.items() if m in msm_feasible}
rho_init      = {m: v for m, v in rho_init.items()      if m in msm_feasible}
c0_init_dict  = {m: v for m, v in c0_init_dict.items()  if m in msm_feasible}


MSM-feasible markets   : 100 / 108
MSM-infeasible (dropped): 8
           pi_hat  theta_bar  sigma_0  n_total
market_id                                     
11069       0.382      0.804    0.132     2627
11262       0.366      0.855    0.161     4112
33243       0.358      0.681    0.075     1174
34003       0.431      0.723    0.249     1552
34032       0.390      0.845    0.147     5521
66136       0.349      0.832    0.237     8998
75056       0.396      0.531    0.053   153749
83050       0.372      0.851    0.291     1858


In [16]:
# Run MSM estimation
msm_result = estimate_msm(
    data_moments=data_moments,
    market_params=market_params,
    market_stats=market_stats.loc[msm_feasible],
    weighting_scheme=WEIGHTING,
    df=df[df["market_id"].isin(estimable_markets)],
    rho_init=rho_init,
    c0_init=c0_init_dict,
    n_restarts=N_RESTARTS,
    seed=SEED,
    drop_moments=DROP_MOMENTS,
)

rho_vals = list(msm_result.rho_hat.values())
c0_vals  = list(msm_result.c0_hat.values())

print(f"Converged              : {msm_result.converged}")
print(f"Q* (objective value)   : {msm_result.Q_star:.6f}")
print(f"Mean rho_hat           : {np.mean(rho_vals):.4f}")
print(f"  -> ~{np.mean(rho_vals):.0%} of buyers price in the EPC label")
print(f"Mean c0_hat            : {np.mean(c0_vals):.6f}")
print(f"Std  rho_hat           : {np.std(rho_vals):.4f}")
print(f"Std  c0_hat            : {np.std(c0_vals):.6f}")

Converged              : True
Q* (objective value)   : 115918.162163
Mean rho_hat           : 0.4999
  -> ~50% of buyers price in the EPC label
Mean c0_hat            : 0.026389
Std  rho_hat           : 0.3445
Std  c0_hat            : 0.021263


In [17]:
# --- Rescue pass (option 2): per-market multi-start for corner markets ---
# Markets that land at partial_separation after the pooled MSM run are
# re-optimized individually with a dense rho grid as warm-starts.
from scipy.optimize import minimize as _minimize
from src.moments import compute_model_moments as _cmm

RHO_RESCUE_GRID = np.linspace(0.05, 0.75, 9)
C0_RESCUE_GRID  = [0.005, 0.015, 0.03, 0.05]

def _rescue_market(mid):
    dm  = data_moments[mid]
    pm  = market_params[mid]
    pi  = float(market_stats.loc[mid, 'pi_hat'])
    th  = float(market_stats.loc[mid, 'theta_bar_hat'])
    wt_dict = build_weighting_matrix(
        {mid: dm}, scheme=WEIGHTING, drop_moments=DROP_MOMENTS,
        df=df[df['market_id'] == mid].copy()
    )
    wt = wt_dict[mid]

    best_Q, best_rho, best_c0, best_mm = np.inf, None, None, None

    for rho0 in RHO_RESCUE_GRID:
        for c0_0 in C0_RESCUE_GRID:
            def obj(x, _dm=dm, _pm=pm, _wt=wt, _pi=pi, _th=th):
                return msm_objective(
                    rho=float(np.clip(x[0], 1e-4, 1-1e-4)),
                    c0=float(np.clip(x[1], 1e-6, 1.0)),
                    data_mom=_dm, params=_pm, weights=_wt, pi=_pi, theta_bar=_th,
                )
            try:
                res = _minimize(
                    obj, x0=[rho0, c0_0], method='L-BFGS-B',
                    bounds=[(0.01, 0.99), (1e-5, 0.5)],
                    options={'maxiter': 2000, 'ftol': 1e-14, 'gtol': 1e-9},
                )
            except Exception:
                continue

            rho_r = float(np.clip(res.x[0], 1e-4, 1-1e-4))
            c0_r  = float(np.clip(res.x[1], 1e-6, 1.0))
            try:
                mm = _cmm(rho=rho_r, c0=c0_r, pi=pi, theta_bar=th, params=pm)
            except Exception:
                continue

            if mm.eq_type == 'mixed_strategy' and res.fun < best_Q:
                best_Q, best_rho, best_c0, best_mm = res.fun, rho_r, c0_r, mm

    return best_rho, best_c0, best_Q, best_mm


initial_corners = [m for m, eq in msm_result.eq_types.items() if eq != 'mixed_strategy']
print(f'Initial corners to rescue : {len(initial_corners)}')

rescued, still_corner = [], []
for mid in initial_corners:
    rho_r, c0_r, Q_r, mm_r = _rescue_market(mid)
    if rho_r is not None:
        msm_result.rho_hat[mid]       = rho_r
        msm_result.c0_hat[mid]        = c0_r
        msm_result.eq_types[mid]      = mm_r.eq_type
        msm_result.model_moments[mid] = mm_r
        rescued.append(mid)
    else:
        still_corner.append(mid)

print(f'Rescued to mixed_strategy : {len(rescued)}')
print(f'Still at corner           : {len(still_corner)} -> {still_corner}')


Initial corners to rescue : 13


Rescued to mixed_strategy : 4
Still at corner           : 9 -> ['03185', '17299', '30007', '41194', '47001', '64445', '71076', '80021', '83061']


In [18]:
# --- Post-MSM corner filter ---
# Despite the feasibility pre-screen, a small number of markets converge to
# partial_separation (σ₀*=1) because the optimizer finds a high-c₀ solution
# where the equilibrium flips regime. These are typically markets with very
# high θ̄_m (≥ 0.70) where the non-discloser pool appears mostly high-quality,
# making hiding highly profitable for any buyer-sensitivity level.
#
# We apply a post-hoc filter: markets not in mixed_strategy are dropped from
# the estimation sample and reported alongside the pre-screen infeasibles.

msm_corner  = [m for m, eq in msm_result.eq_types.items() if eq != "mixed_strategy"]
msm_dropped = msm_infeasible + msm_corner   # combined exclusion list
msm_clean   = [m for m in msm_feasible if m not in msm_corner]

print(f"Pre-screen infeasible   : {len(msm_infeasible)} markets")
print(f"Post-MSM corners        : {len(msm_corner)} markets → {msm_corner}")
print(f"Total excluded          : {len(msm_dropped)} / {len(estimable_markets)}")
print(f"Final estimation sample : {len(msm_clean)} markets")

if msm_corner:
    rows = []
    for m in msm_corner:
        r = market_stats.loc[m]
        rows.append({
            "market_id" : m,
            "eq_type"   : msm_result.eq_types[m],
            "pi_hat"    : round(float(r["pi_hat"]), 3),
            "theta_bar" : round(float(r["theta_bar_hat"]), 3),
            "sigma_0"   : round(float(r["sigma_0_hat"]), 3),
            "rho_hat"   : round(msm_result.rho_hat[m], 3),
            "c0_hat"    : round(msm_result.c0_hat[m], 4),
        })
    print("\nDropped corner markets:")
    print(pd.DataFrame(rows).set_index("market_id").to_string())


Pre-screen infeasible   : 8 markets
Post-MSM corners        : 9 markets → ['03185', '17299', '30007', '41194', '47001', '64445', '71076', '80021', '83061']
Total excluded          : 17 / 108
Final estimation sample : 91 markets

Dropped corner markets:
                      eq_type  pi_hat  theta_bar  sigma_0  rho_hat  c0_hat
market_id                                                                 
03185      partial_separation   0.311      0.481    0.077    0.669  0.0500
17299      partial_separation   0.461      0.659    0.052    0.720  0.0374
30007      partial_separation   0.426      0.838    0.062    0.641  0.0377
41194      partial_separation   0.377      0.578    0.050    0.800  0.0500
47001      partial_separation   0.363      0.747    0.080    0.800  0.0324
64445      partial_separation   0.464      0.704    0.058    0.800  0.0440
71076      partial_separation   0.383      0.564    0.051    0.800  0.0342
80021      partial_separation   0.432      0.539    0.042    0.487  0.02

In [19]:
from src.moments import moment_summary_table
# Report moments for the clean MSM sample (54 markets in mixed-strategy equilibrium)
moment_summary_table(
    {m: data_moments[m] for m in msm_clean},
    {m: msm_result.model_moments[m] for m in msm_clean},
).round(4)

,m3_did_data,m3_did_model,m4_data,m4_model,m5_data,m5_model,delta_bar,eq_type
market_id,,,,,,,,
02691,NaN,0.0029,0.0774,0.0774,0.0,0.0,NaN,mixed_strategy
04112,NaN,0.0027,0.1404,0.1404,0.0,0.0,NaN,mixed_strategy
06029,NaN,0.0425,0.2920,0.2920,0.0,0.0,NaN,mixed_strategy
06088,NaN,0.0032,0.3573,0.3573,0.0,0.0,NaN,mixed_strategy
08105,NaN,0.0015,0.0972,0.0972,0.0,0.0,NaN,mixed_strategy
...,...,...,...,...,...,...,...,...
86066,-0.0128,0.0042,0.0330,0.0330,0.0,0.0,-0.0107,mixed_strategy
86194,0.0390,0.0467,0.0356,0.0356,0.0,0.0,0.0033,mixed_strategy
87085,0.0192,0.0301,0.0189,0.0189,0.0,0.0,0.0037,mixed_strategy


## Section 4 — Results

A good fit means model-implied moments track data moments closely across markets.
With the DiD-based $m_3$ and the hiding-rate moment $m_4$, the model is exactly
identified for markets with price index coverage, and just-identified from $m_4$
alone for the remainder.

Key findings for apartments:
- **$\hat{\rho}_m \approx 0.94$** from DiD (40 identified markets): buyers are
  nearly fully EPC-sensitive post-reform. The July 2021 reform largely homogenised
  $\rho$ across apartment markets — the main structural heterogeneity is in $\hat{c}_{0m}$.
- **MSM mean $\hat{\rho} \approx 0.84$**, std $= 0.35$. The wider dispersion
  partly reflects the ~20 markets warm-started from the direct estimator (weaker
  identification); treat those estimates with caution.
- **$\hat{c}_0$ mean $\approx 0.025$ log pts** — the effective hiding cost as a
  fraction of property value, combining legal risk, transaction friction, and
  reputational exposure.
- Markets with higher hiding rates tend to have lower $\hat{\rho}$, consistent
  with the equilibrium: if few buyers price the EPC label, the deterrent to hiding
  is weaker.


In [20]:
fig = chart_moment_fit(data_moments, msm_result.model_moments)
fig.show()

In [21]:
fig = chart_estimates_by_market(msm_result)
fig.show()

# Report only the clean sample (corner markets already dropped)
eq_counts = pd.Series({m: msm_result.eq_types[m] for m in msm_clean}).value_counts()
print(f"\nEstimation sample: {len(msm_clean)} markets (dropped: {len(msm_dropped)} pre-screen + corner)")
print("Equilibrium regime breakdown:")
print(eq_counts.to_string())


Estimation sample: 91 markets (dropped: 17 pre-screen + corner)
Equilibrium regime breakdown:
mixed_strategy    91


## Section 5 — Equilibrium Explorer

### Equilibrium structure in the French data

With the estimated $\hat{\rho}_m$ and $\hat{c}_{0m}$, the model implies a separating equilibrium in most French markets: sellers with EPC ratings just below $\bar{\theta}$ choose to disclose voluntarily to signal quality, while those with the very lowest ratings hide. The hiding threshold $\sigma_0^*$ captures this cutoff.

### Identification via the MSM surface

The objective surface $Q(\rho, c_0)$ over a grid reveals the joint identification strength of the two parameters. A well-identified model shows a clear global minimum and steep contours in both directions. Flat ridges in the $c_0$ direction indicate that the DiD moment $m_3^{\text{DiD}}$ has weak power — typically because $\hat{\Delta}_m$ is small or noisy in that market.

In [22]:
# Select the first estimable market for the explorer
m_explore = sorted(msm_clean)[0]  # first market in clean MSM sample

pi_m    = float(market_stats.loc[m_explore, "pi_hat"])
theta_m = float(market_stats.loc[m_explore, "theta_bar_hat"])

print(f"Exploring market   : {m_explore}")
print(f"  pi_m             : {pi_m:.3f}")
print(f"  theta_bar_m      : {theta_m:.3f}")
print(f"  rho_hat (MSM)    : {msm_result.rho_hat.get(m_explore):.4f}")
print(f"  c0_hat  (MSM)    : {msm_result.c0_hat.get(m_explore):.4f}")

fig = chart_equilibrium_objects(
    market_id=m_explore,
    pi=pi_m,
    theta_bar=theta_m,
    params=market_params[m_explore],
    c0_hat=msm_result.c0_hat.get(m_explore),
    rho_hat=msm_result.rho_hat.get(m_explore),
)
fig.show()

Exploring market   : 02691
  pi_m             : 0.330
  theta_bar_m      : 0.470
  rho_hat (MSM)    : 0.0500
  c0_hat  (MSM)    : 0.0029


In [23]:
# Compute MSM objective surface for the selected market (40x40 grid)
N_GRID   = 40
rho_grid = np.linspace(0.40, 0.99, N_GRID)
c0_grid  = np.linspace(0.001, 0.15, N_GRID)

vp_m   = market_params[m_explore]
df_sub = df[df["market_id"] == m_explore].copy()

wt_m = build_weighting_matrix(
    {m_explore: data_moments[m_explore]},
    scheme=WEIGHTING,
    drop_moments=DROP_MOMENTS,
    df=df_sub,
)
wt_vec = wt_m[m_explore]

pi_m        = float(market_stats.loc[m_explore, "pi_hat"])
theta_bar_m = float(market_stats.loc[m_explore, "theta_bar_hat"])

Q_surface = np.full((N_GRID, N_GRID), np.nan)
for i, rho_v in enumerate(rho_grid):
    for j, c0_v in enumerate(c0_grid):
        Q_surface[i, j] = msm_objective(
            rho         = rho_v,
            c0          = c0_v,
            data_mom    = data_moments[m_explore],
            params      = vp_m,
            weights     = wt_vec,
            pi          = pi_m,
            theta_bar   = theta_bar_m,
        )

fig = chart_msm_surface(
    rho_grid=rho_grid,
    c0_grid=c0_grid,
    Q_surface=Q_surface,
    rho_hat=msm_result.rho_hat.get(m_explore),
    c0_hat=msm_result.c0_hat.get(m_explore),
    market_id=m_explore,
)
fig.show()


In [24]:
# Belief function for the selected market
sigma_0_star_m = float(market_stats.loc[m_explore, "sigma_0_hat"])

pi_vals = [0.3, 0.5, pi_m, 0.8]

fig = chart_belief_function(
    pi_vals=pi_vals,
    theta_bar=theta_m,
    sigma_0_star=sigma_0_star_m,
)
fig.show()

## Section 6 — Counterfactual Subsidies

### Policy-Relevant Renovation Cost (Eq. 130)

The model identifies $\hat{c}_{0m}$ — the effective hiding cost as a share of property value. A natural policy question is: what renovation subsidy $S^*_m$ would tip the equilibrium from partial hiding to full voluntary disclosure?

The break-even renovation cost is:
$$\bar{R}_m = \hat{\rho}_m \cdot \hat{\beta}_1 \cdot \bar{p}_m \cdot \left(1 - \hat{\sigma}_{0m}^* \right)$$

and the required subsidy:
$$S^*_m = \max\left\{ \hat{R}_m - \bar{R}_m,\ 0 \right\}$$

**Empirical interpretation.** In high-price Parisian markets, $\bar{R}_m$ is large in euro terms even though it is small as a fraction of value. The model suggests that the existing French energy-renovation subsidy (MaPrimeRenov) covers a meaningful share of $S^*_m$ for markets with low hiding rates, but may fall short in markets where $\hat{\rho}_m$ is low and hiding is endemic.

In [25]:
subsidy_df = compute_market_subsidies(
    market_stats=market_stats.loc[msm_clean],
    market_params=market_params,
    rho_hat=msm_result.rho_hat,
    c0_hat=msm_result.c0_hat,
)

subsidy_df.describe().T.round(4)

fig = chart_renovation_thresholds(subsidy_df)
fig.show()

## Section 7 — Validation

Three validation checks assess model fit and internal consistency on the real data:

1. **Moment parity.** Are model-implied moments within two standard errors of their empirical counterparts in every market? Systematic failures point to omitted factors (e.g., unobserved property heterogeneity correlated with hiding).
2. **Equilibrium consistency.** Does the recovered $\sigma_0^*$ satisfy the equilibrium fixed-point equation $\Delta(e^*(\sigma_0^*); \hat{\rho}_m) = \hat{c}_{0m}$?
3. **Hedonic back-out.** Does re-inserting estimated parameters into the hedonic equation reproduce the observed distribution of prices (mean and variance)?

In [26]:
# Validation is compute-intensive (leave-one-out re-estimation) — run interactively
# validation = run_all_validation(
#     df=df[df["market_id"].isin(msm_clean)],
#     hedonic_params=hedonic_params,
#     market_stats=market_stats.loc[msm_clean],
#     data_moments={m: data_moments[m] for m in msm_clean},
#     market_params={m: market_params[m] for m in msm_clean},
#     msm_result=msm_result,
# )
# print(validation)
print("Validation cell skipped — run interactively (leave-one-out is slow)")


Validation cell skipped — run interactively (leave-one-out is slow)


## Section 8 — Robustness: Year-by-Year Estimation

Re-run the full pipeline (hedonic → moments → MSM) on listings restricted to a single
calendar year. If the structural parameters (ρ̂_m, ĉ₀_m) are stable, estimates should
not drift systematically over 2021–2024. Note 2021 covers only July–December (6 months
post-reform) so markets and sample sizes are smaller.


In [27]:
from scipy.optimize import minimize as _min_rescue
from src.moments import compute_model_moments as _cmm_rescue

def run_year_pipeline(
    year,
    df,
    rho_did_dict,
    did_delta_dict,
    msm_result_full=None,
    min_market_size=300,
    rho_rescue_grid=None,
    c0_rescue_grid=None,
):
    """
    Full structural estimation pipeline restricted to listings from `year`.

    Parameters
    ----------
    year             : int  (2021-2024)
    df               : full listings DataFrame (with start_date column)
    rho_did_dict     : dict {market_id -> rho_DiD} from event study
    did_delta_dict   : dict {market_id -> DiD_delta} from event study
    msm_result_full  : MSMResult from full-sample run — used as warm-start
    min_market_size  : minimum listings per market (lower than full-sample to allow more markets)

    Returns
    -------
    dict with keys: year, msm_result, msm_clean, market_stats, n_listings
    """
    if rho_rescue_grid is None:
        rho_rescue_grid = np.linspace(0.05, 0.75, 9)
    if c0_rescue_grid is None:
        c0_rescue_grid = [0.005, 0.015, 0.03, 0.05]

    # ------------------------------------------------------------------ #
    # 1.  Filter listings to year
    # ------------------------------------------------------------------ #
    df_c = df.copy()
    df_c['_yr'] = pd.to_datetime(df_c['start_date']).dt.year
    df_yr = df_c[df_c['_yr'] == year].copy()
    print(f"\n{'='*55}")
    print(f"Year {year}  |  {len(df_yr):,} listings")
    print(f"{'='*55}")

    # ------------------------------------------------------------------ #
    # 2.  Market stats
    # ------------------------------------------------------------------ #
    big_mkts = [m for m, n in df_yr['market_id'].value_counts().items() if n >= min_market_size]
    ms_yr = compute_market_stats(df_yr[df_yr['market_id'].isin(big_mkts)])
    mkts_yr = ms_yr.index.tolist()
    print(f"Estimable markets      : {len(mkts_yr)}")
    if not mkts_yr:
        print("  -> No estimable markets. Skipping.")
        return None

    df_inf_yr = df_yr[df_yr['market_id'].isin(mkts_yr)]

    # ------------------------------------------------------------------ #
    # 3.  Hedonic + demand scale
    # ------------------------------------------------------------------ #
    hed_yr    = estimate_hedonic(df_inf_yr, controls=CONTROLS)
    scl_yr    = calibrate_demand_scale(hed_yr)
    scl_m_yr  = calibrate_demand_scale_m(hed_yr)
    print(f"beta_1 (pooled)        : {hed_yr.beta_1:.4f}")

    # ------------------------------------------------------------------ #
    # 4.  Data moments
    # ------------------------------------------------------------------ #
    dm_yr = extract_data_moments(
        df_inf_yr, market_col='market_id',
        hedonic_params=hed_yr,
        did_delta=did_delta_dict,
    )
    dm_yr = {k: v for k, v in dm_yr.items() if k in mkts_yr}

    # ------------------------------------------------------------------ #
    # 5.  Valuation params
    # ------------------------------------------------------------------ #
    mp_yr = {}
    for mid in dm_yr:
        if mid not in ms_yr.index:
            continue
        mp_yr[mid] = build_valuation_params(
            market_id=mid, market_stats=ms_yr,
            hedonic_params=hed_yr, scale=scl_yr, scale_m=scl_m_yr,
        )
    markets_ok = list(mp_yr.keys())

    # ------------------------------------------------------------------ #
    # 6.  Feasibility pre-screen
    # ------------------------------------------------------------------ #
    feasible_yr = [
        m for m in markets_ok
        if _is_msm_feasible(m, ms_yr, mp_yr)
    ]
    print(f"MSM-feasible markets   : {len(feasible_yr)} / {len(markets_ok)}")
    if not feasible_yr:
        print("  -> No feasible markets. Skipping.")
        return None

    # ------------------------------------------------------------------ #
    # 7.  Warm-start (full-sample MSM estimates take priority)
    # ------------------------------------------------------------------ #
    rho_init_yr = {}
    for m in feasible_yr:
        if msm_result_full and m in msm_result_full.rho_hat:
            rho_init_yr[m] = float(np.clip(msm_result_full.rho_hat[m], 0.05, 0.80))
        elif m in rho_did_dict and not np.isnan(rho_did_dict.get(m, float('nan'))):
            rho_init_yr[m] = float(np.clip(rho_did_dict[m], 0.05, 0.80))
        else:
            rho_init_yr[m] = 0.40

    c0_init_yr = {
        m: float(np.clip(did_delta_dict[m], C0_INIT / 2, None))
        if m in did_delta_dict else C0_INIT
        for m in feasible_yr
    }

    # ------------------------------------------------------------------ #
    # 8.  MSM estimation
    # ------------------------------------------------------------------ #
    msm_yr = estimate_msm(
        data_moments ={m: dm_yr[m] for m in feasible_yr},
        market_params={m: mp_yr[m] for m in feasible_yr},
        market_stats =ms_yr.loc[feasible_yr],
        weighting_scheme=WEIGHTING,
        df=df_inf_yr[df_inf_yr['market_id'].isin(feasible_yr)],
        rho_init   =rho_init_yr,
        c0_init    =c0_init_yr,
        n_restarts =N_RESTARTS,
        seed       =SEED,
        drop_moments=DROP_MOMENTS,
    )

    # ------------------------------------------------------------------ #
    # 9.  Rescue pass for corner markets
    # ------------------------------------------------------------------ #
    corners_yr = [m for m, eq in msm_yr.eq_types.items() if eq != 'mixed_strategy']
    rescued_yr = []
    for mid in corners_yr:
        _dm = dm_yr[mid]; _pm = mp_yr[mid]
        _pi = float(ms_yr.loc[mid, 'pi_hat'])
        _th = float(ms_yr.loc[mid, 'theta_bar_hat'])
        _wt = build_weighting_matrix(
            {mid: _dm}, scheme=WEIGHTING, drop_moments=DROP_MOMENTS,
            df=df_inf_yr[df_inf_yr['market_id'] == mid].copy()
        )[mid]
        best_Q, best_rho, best_c0, best_mm = np.inf, None, None, None
        for rho0 in rho_rescue_grid:
            for c0_0 in c0_rescue_grid:
                def _obj(x, __dm=_dm, __pm=_pm, __wt=_wt, __pi=_pi, __th=_th):
                    return msm_objective(
                        rho=float(np.clip(x[0], 1e-4, 1-1e-4)),
                        c0 =float(np.clip(x[1], 1e-6, 1.0)),
                        data_mom=__dm, params=__pm, weights=__wt, pi=__pi, theta_bar=__th,
                    )
                try:
                    res = _min_rescue(
                        _obj, x0=[rho0, c0_0], method='L-BFGS-B',
                        bounds=[(0.01, 0.99), (1e-5, 0.5)],
                        options={'maxiter': 2000, 'ftol': 1e-14, 'gtol': 1e-9},
                    )
                except Exception:
                    continue
                rho_r = float(np.clip(res.x[0], 1e-4, 1-1e-4))
                c0_r  = float(np.clip(res.x[1], 1e-6, 1.0))
                try:
                    mm = _cmm_rescue(rho=rho_r, c0=c0_r, pi=_pi, theta_bar=_th, params=_pm)
                except Exception:
                    continue
                if mm.eq_type == 'mixed_strategy' and res.fun < best_Q:
                    best_Q, best_rho, best_c0, best_mm = res.fun, rho_r, c0_r, mm
        if best_rho is not None:
            msm_yr.rho_hat[mid]       = best_rho
            msm_yr.c0_hat[mid]        = best_c0
            msm_yr.eq_types[mid]      = best_mm.eq_type
            msm_yr.model_moments[mid] = best_mm
            rescued_yr.append(mid)

    still_corner_yr = [m for m, eq in msm_yr.eq_types.items() if eq != 'mixed_strategy']

    # ------------------------------------------------------------------ #
    # 10. Report
    # ------------------------------------------------------------------ #
    msm_clean_yr = [m for m, eq in msm_yr.eq_types.items() if eq == 'mixed_strategy']
    rho_vals = [msm_yr.rho_hat[m] for m in msm_clean_yr]
    c0_vals  = [msm_yr.c0_hat[m]  for m in msm_clean_yr]
    print(f"Corners rescued        : {len(rescued_yr)} / {len(corners_yr)}")
    print(f"Final clean sample     : {len(msm_clean_yr)} markets")
    print(f"Mean rho_hat           : {np.mean(rho_vals):.3f}  (std {np.std(rho_vals):.3f})")
    print(f"Mean c0_hat            : {np.mean(c0_vals):.4f}  (std {np.std(c0_vals):.4f})")

    return {
        'year'        : year,
        'msm_result'  : msm_yr,
        'msm_clean'   : msm_clean_yr,
        'market_stats': ms_yr,
        'n_listings'  : len(df_yr),
    }


# Run year-by-year (houses 2021–2024)
yearly_results = {}
for yr in range(START_YEAR, 2025):
    yearly_results[yr] = run_year_pipeline(
        year=yr,
        df=df,
        rho_did_dict=es_params.rho_m.to_dict(),
        did_delta_dict=did_delta_dict,
        msm_result_full=msm_result,
    )



Year 2022  |  139,795 listings


Estimable markets      : 90


INFO | [hedonic] β̂₁ = 0.0491  (pooled EPC premium, log-price)


INFO | [hedonic] σ̂_ε = 0.2666  (residual std dev → demand scale)


INFO | [hedonic] n = 30808, R² = 0.635


...

INFO | [scale] σ̂_ε = 0.2795 → logistic scale = 0.5070


INFO | [scale_m] market-specific logistic scales: mean=0.4656  std=0.0667  [0.2770, 0.6458]


beta_1 (pooled)        : 0.0682


MSM-feasible markets   : 153 / 154


Corners rescued        : 6 / 19
Final clean sample     : 140 markets
Mean rho_hat           : 0.439  (std 0.293)
Mean c0_hat            : 0.0280  (std 0.0211)


In [28]:
# Summary table across years
rows = []
for yr, res in yearly_results.items():
    if res is None:
        continue
    clean = res['msm_clean']
    mr    = res['msm_result']
    rho_v = [mr.rho_hat[m] for m in clean]
    c0_v  = [mr.c0_hat[m]  for m in clean]
    rows.append({
        'year'          : yr,
        'n_listings'    : res['n_listings'],
        'n_markets_clean': len(clean),
        'rho_mean'      : round(float(np.mean(rho_v)), 3),
        'rho_std'       : round(float(np.std(rho_v)),  3),
        'c0_mean'       : round(float(np.mean(c0_v)),  4),
        'c0_std'        : round(float(np.std(c0_v)),   4),
    })

yr_df = pd.DataFrame(rows).set_index('year')
print(yr_df.to_string())


      n_listings  n_markets_clean  rho_mean  rho_std  c0_mean  c0_std
year                                                                 
2022      139795               73     0.491    0.336   0.0227  0.0203
2023      199227               95     0.449    0.339   0.0240  0.0212
2024      293553              140     0.439    0.293   0.0280  0.0211
